In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads

df = pd.read_csv('src/1857_commercial_guide_20250220.csv')
print(len(df))
df = df[df['geometry'].notna()]
print(len(df))
gdf = gpd.GeoDataFrame(df, geometry= df['geometry'].apply(loads)).set_geometry('geometry').set_crs('EPSG:4326')
gdf.to_file('src/1857_commercial_guide_20250507.geojson', driver='GeoJSON')

5482
5458


TypeError: to_file() missing 1 required positional argument: 'filename'

In [55]:
import pandas as pd

def keep_only_numerical_value(x):
    if isinstance(x, str):
        return ''.join(filter(str.isdigit, x))
    return x

common_cols = ['OCC_COMPL', 'LAST_N', 'FIRST_N', 'PER_GRP', 'PER_COMPL',
       'LOC_PAR', 'LOC_STR']
df_old = pd.read_csv('src/1857_commercial_guide_20250220.csv')
dfp = pd.read_csv('src/1857_cleaned_olivier.csv')

dfp['PAGE_NUM'] = dfp['PAGE_NUM'].ffill().apply(keep_only_numerical_value)
dfp['match_string'] = dfp[common_cols].apply(lambda x: ' '.join(x.astype(str)), axis=1)
df_old['match_string'] = df_old[common_cols].apply(lambda x: ' '.join(x.astype(str)), axis=1)

In [ ]:
match_string_to_page = {
    "nan Gariboldi Marco nan fu Sante, s. M. Gloriosa dei Frari campo s. Polo": 140, #ou 141
}

dfm = df_old.merge(dfp[['PAGE_NUM', 'match_string']], on='match_string', how='left')
for k, v in match_string_to_page.items():
    dfm.loc[dfm['match_string'] == k, 'PAGE_NUM'] = v

dfm.drop_duplicates().reset_index().drop(columns='index').to_csv('src/1857_commercial_guide_20250624.csv', index=True)